# Confronto paesi per classi dimensionali ufficiali

            Questo foglio usa il dataset pulito Eurostat SBS per confrontare i paesi europei del perimetro configurato sulle classi 0-9, 10-19, 20-49, 50-249 e 250+.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from valore_aggiunto_imprese.analysis import (
    METRIC_LABELS,
    add_source_note,
    latest_common_year_with_values,
    latest_year_with_values,
    SIZE_ORDER_BUSINESS,
    SIZE_ORDER_OFFICIAL,
    add_share,
    enrich_business_demography,
    enrich_sbs,
    read_project_csv,
)

pd.options.display.max_columns = 80
pd.options.display.float_format = "{:,.2f}".format

## Dataset analitico

In [ ]:
sbs = enrich_sbs(read_project_csv("data/processed/official_eurostat_sbs.csv"))
latest_year = latest_year_with_values(sbs, metric_code="AV_MEUR")
sbs[["country_label", "year", "sector_label", "size_label", "metric_code", "value"]].head()

## Quote di valore aggiunto per classe

In [ ]:
va = (
    sbs.query("metric_code == 'AV_MEUR' and year == @latest_year")
    .dropna(subset=["value"])
    .groupby(["country_label", "size_label"], observed=True, as_index=False)["value"]
    .sum()
)
va["size_label"] = pd.Categorical(va["size_label"], SIZE_ORDER_OFFICIAL, ordered=True)
va_share = add_share(va, ["country_label"])

fig = px.bar(
    va_share.sort_values(["country_label", "size_label"]),
    x="country_label",
    y="share_pct",
    color="size_label",
    category_orders={"size_label": SIZE_ORDER_OFFICIAL},
    labels={"country_label": "Paese", "share_pct": "% valore aggiunto", "size_label": "Classe"},
    title=f"Quote di valore aggiunto per classe dimensionale - {latest_year}",
)
add_source_note(fig, "Eurostat Structural Business Statistics")
fig

## Peso numerico e peso economico

In [ ]:
comparison_year = latest_common_year_with_values(sbs, ["AV_MEUR", "ENT_NR"])
comparison = (
    sbs.query("year == @comparison_year and metric_code in ['AV_MEUR', 'ENT_NR']")
    .dropna(subset=["value"])
    .groupby(["country_label", "size_label", "metric_code"], observed=True, as_index=False)["value"]
    .sum()
)
comparison_share = add_share(comparison, ["country_label", "metric_code"])
comparison_wide = comparison_share.pivot_table(
    index=["country_label", "size_label"],
    columns="metric_code",
    values="share_pct",
    aggfunc="sum",
    observed=True,
).reset_index()

fig = px.scatter(
    comparison_wide,
    x="ENT_NR",
    y="AV_MEUR",
    color="size_label",
    hover_name="country_label",
    category_orders={"size_label": SIZE_ORDER_OFFICIAL},
    labels={"ENT_NR": "% imprese", "AV_MEUR": "% valore aggiunto", "size_label": "Classe"},
    title=f"Peso numerico e peso economico delle classi - {comparison_year}",
)
add_source_note(fig, "Eurostat Structural Business Statistics")
fig

## Produttivita apparente per classe

In [ ]:
productivity_year = latest_year_with_values(sbs, metric_code="LABPRY_TEUR")
productivity = (
    sbs.query("metric_code == 'LABPRY_TEUR' and year == @productivity_year")
    .dropna(subset=["value"])
    .groupby(["country_label", "size_label"], observed=True, as_index=False)["value"]
    .mean()
)
productivity["size_label"] = pd.Categorical(
    productivity["size_label"], SIZE_ORDER_OFFICIAL, ordered=True
)

fig = px.box(
    productivity,
    x="size_label",
    y="value",
    points="all",
    color="size_label",
    category_orders={"size_label": SIZE_ORDER_OFFICIAL},
    labels={"size_label": "Classe", "value": "Migliaia di euro"},
    title=f"Distribuzione della produttivita apparente - {productivity_year}",
)
add_source_note(fig, "Eurostat Structural Business Statistics")
fig

## Ranking sintetico

In [ ]:
(
    va_share.pivot_table(
        index="country_label",
        columns="size_label",
        values="share_pct",
        aggfunc="sum",
        observed=True,
    )
    .reindex(columns=SIZE_ORDER_OFFICIAL)
    .round(1)
    .sort_values("250+", ascending=False)
)